In [1]:
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer


# ==========================================
# 1. Knowledge Base
# ==========================================

documents = [
    {"id": "HR-001", "text": "Employees receive 26 annual leave days."},
    {"id": "HR-002", "text": "Leave requests must be approved by the manager."},
    {"id": "HR-003", "text": "Employees have supplementary health insurance."},
    {"id": "HR-004", "text": "Monthly salaries are paid at the end of each month."},
    {"id": "HR-005", "text": "Employees can work remotely two days per week."},
    {"id": "HR-006", "text": "Employees can request unpaid leave."},
]

document_texts = [document["text"] for document in documents]


# ==========================================
# 2. Keyword Search - TF-IDF
# ==========================================

tfidf_vectorizer = TfidfVectorizer()

tfidf_document_vectors = tfidf_vectorizer.fit_transform(document_texts)


def keyword_search(query, top_k=3):

    query_vector = tfidf_vectorizer.transform([query])

    scores = cosine_similarity(query_vector, tfidf_document_vectors)[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        if scores[index] <= 0:
            continue

        results.append(
            {
                "id": documents[index]["id"],
                "text": documents[index]["text"],
                "score": float(scores[index]),
            }
        )

    return results


# ==========================================
# 3. Semantic Search - Embeddings
# ==========================================

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

document_embeddings = embedding_model.encode(
    document_texts, convert_to_numpy=True, normalize_embeddings=True
)


def semantic_search(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append(
            {
                "id": documents[index]["id"],
                "text": documents[index]["text"],
                "score": float(scores[index]),
            }
        )

    return results


# ==========================================
# 4. Compare Both Search Methods
# ==========================================

queries = ["annual leave days", "How much time off can employees take each year?"]

for query in queries:
    print("\n" + "=" * 70)
    print("QUERY:", query)

    # Keyword Search
    keyword_results = keyword_search(query)

    print("\nKEYWORD SEARCH RESULTS:")

    for result in keyword_results:
        print(f"{result['id']} | Score: {result['score']:.4f}")
        print(result["text"])

    # Semantic Search
    semantic_results = semantic_search(query)

    print("\nSEMANTIC SEARCH RESULTS:")

    for result in semantic_results:
        print(f"{result['id']} | Score: {result['score']:.4f}")
        print(result["text"])

e:\AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 907.89it/s]



QUERY: annual leave days

KEYWORD SEARCH RESULTS:
HR-001 | Score: 0.6912
Employees receive 26 annual leave days.
HR-005 | Score: 0.1771
Employees can work remotely two days per week.
HR-006 | Score: 0.1746
Employees can request unpaid leave.

SEMANTIC SEARCH RESULTS:
HR-001 | Score: 0.7056
Employees receive 26 annual leave days.
HR-006 | Score: 0.5127
Employees can request unpaid leave.
HR-004 | Score: 0.4362
Monthly salaries are paid at the end of each month.

QUERY: How much time off can employees take each year?

KEYWORD SEARCH RESULTS:
HR-006 | Score: 0.3846
Employees can request unpaid leave.
HR-005 | Score: 0.2782
Employees can work remotely two days per week.
HR-004 | Score: 0.2260
Monthly salaries are paid at the end of each month.

SEMANTIC SEARCH RESULTS:
HR-001 | Score: 0.6715
Employees receive 26 annual leave days.
HR-005 | Score: 0.5735
Employees can work remotely two days per week.
HR-006 | Score: 0.5719
Employees can request unpaid leave.
